In [1]:
from pyspark.sql import SparkSession

# Membuat SparkSession — "local[*]" berarti gunakan seluruh core CPU yang tersedia di VM
spark = SparkSession.builder \
    .appName("Pertemuan4-PengenalanPySpark") \
    .master("local[*]") \
    .getOrCreate()

# Mengurangi banyaknya pesan log teknis agar output lebih bersih
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession berhasil dibuat!")
print("Versi Spark:", spark.version)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/16 08:15:54 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession berhasil dibuat!
Versi Spark: 3.5.9


In [2]:
# Sel ini membuat dataset baru untuk Tugas Mandiri Pertemuan 4 dan mengunggahnya ke HDFS
import numpy as np
import pandas as pd

np.random.seed(99)
n = 1000
kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga", "Olahraga"]
kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo", "Kebumen"]
metode_bayar_list = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]
tanggal_range = pd.date_range("2026-09-01", "2026-09-30", freq="D")

data = {
    "order_id": [f"ORD-{3000 + i}" for i in range(n)],
    "tanggal": np.random.choice(tanggal_range, size=n).astype(str),
    "kategori": np.random.choice(kategori_list, size=n),
    "kota": np.random.choice(kota_list, size=n),
    "unit_terjual": np.random.randint(1, 12, size=n),
    "harga_satuan": np.random.choice([20000, 45000, 60000, 90000, 125000, 200000, 350000], size=n),
    "metode_pembayaran": np.random.choice(metode_bayar_list, size=n),
    "rating": np.random.choice([1, 2, 3, 4, 5, np.nan], size=n, p=[0.03, 0.02, 0.10, 0.30, 0.35, 0.20]),
}
df_tugas4 = pd.DataFrame(data)
df_tugas4.to_csv("transaksi_september_2026.csv", index=False)
print(f"Dataset dibuat: {df_tugas4.shape[0]} baris")

# Mengunggah ke HDFS
!hdfs dfs -mkdir -p /user/mahasiswa/tugas4
!hdfs dfs -put -f transaksi_september_2026.csv /user/mahasiswa/tugas4/
print("Berhasil diunggah ke HDFS: /user/mahasiswa/tugas4/transaksi_september_2026.csv")

Dataset dibuat: 1000 baris
Berhasil diunggah ke HDFS: /user/mahasiswa/tugas4/transaksi_september_2026.csv


In [3]:
#A. 

#Membaca dataset
df = spark.read.csv("transaksi_september_2026.csv", header=True, inferSchema=True)

#Menampilkan printSchema
print("Menampilkan printSchema")
df.printSchema()

#Menampilkan 10 baris pertama 
print("\nMenampilkan 10 baris pertama")
df.show(10)

#Menghitung jumlah baris 
print("\nMenampilkan jumlah baris")
print("Jumlah baris:", df.count())

Menampilkan printSchema
root
 |-- order_id: string (nullable = true)
 |-- tanggal: timestamp (nullable = true)
 |-- kategori: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- metode_pembayaran: string (nullable = true)
 |-- rating: double (nullable = true)


Menampilkan 10 baris pertama
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|order_id|            tanggal|            kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|rating|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|ORD-3000|2026-09-02 00:00:00|        Rumah Tangga|Yogyakarta|           3|       90000|              COD|   4.0|
|ORD-3001|2026-09-04 00:00:00|   Makanan & Minuman|      Solo|           3|      200000|         E-Wallet|   5.0|
|ORD-3002|2026-09-26 00:00:00|Keseh

In [4]:
## B. MENANGANI DATA KOSONG ##
#(1) Sebelum menggunakan penanganan
from pyspark.sql.functions import col

kolom_kosong = df.filter(col("rating").isNull()).count()
print("Jumlah nilai rating kosong:", kolom_kosong)

#(2) Menggunakan df.na.fill()
df = df.na.fill({"rating": 4})

kolom_kosong = df.filter(col("rating").isNull()).count()
print("Jumlah nilai rating kosong:", kolom_kosong)

Jumlah nilai rating kosong: 204
Jumlah nilai rating kosong: 0


In [5]:
## C. TRANSFORMASI DATA ##
from pyspark.sql.functions import col, when

#Menambahkan kolom total_pendapatan dan tier_transaksi
df = df.withColumn("total_pendapatan", col("unit_terjual") * col("harga_satuan"))
df = df.withColumn("tier_transaksi", when(col("total_pendapatan") > 500000, "Besar").otherwise("Kecil"))

df.show()

+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+----------------+--------------+
|order_id|            tanggal|            kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|rating|total_pendapatan|tier_transaksi|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+----------------+--------------+
|ORD-3000|2026-09-02 00:00:00|        Rumah Tangga|Yogyakarta|           3|       90000|              COD|   4.0|          270000|         Kecil|
|ORD-3001|2026-09-04 00:00:00|   Makanan & Minuman|      Solo|           3|      200000|         E-Wallet|   5.0|          600000|         Besar|
|ORD-3002|2026-09-26 00:00:00|Kesehatan & Kecan...|  Semarang|           8|       60000|         E-Wallet|   3.0|          480000|         Kecil|
|ORD-3003|2026-09-09 00:00:00|   Makanan & Minuman|  Semarang|           6|      350000|    Transfer Bank|   4.0|         21

In [6]:
## D. ANALISIS DENGAN GROUPBY ##
from pyspark.sql.functions import sum as spark_sum, avg

kategori_pendapatan = df.groupBy("kategori").agg(spark_sum("total_pendapatan").alias("total_pendapatan")).orderBy(col("total_pendapatan").desc())
kota_tier = df.filter(col("tier_transaksi") == "Besar").groupBy("kota").count().withColumnRenamed("count", "jumlah_transaksi").orderBy(col("jumlah_transaksi").desc())
rating_pembayaran = df.groupBy("metode_pembayaran").agg(avg("rating").alias("avg rating")).orderBy(col("avg rating").desc())

print("(1) Menampilkan kategori dengan total_pendapatan tertinggi:")
kategori_pendapatan.show(1)

print("(2) Menampilkan kota dengan jumlah transaksi tier 'Besar' terbanyak:")
kota_tier.show(1)

print("(3) Rata-rata rating pada masing-masing metode_pembayaran:")
rating_pembayaran.show()

(1) Menampilkan kategori dengan total_pendapatan tertinggi:
+------------+----------------+
|    kategori|total_pendapatan|
+------------+----------------+
|Rumah Tangga|       138665000|
+------------+----------------+
only showing top 1 row

(2) Menampilkan kota dengan jumlah transaksi tier 'Besar' terbanyak:
+----+----------------+
|kota|jumlah_transaksi|
+----+----------------+
|Solo|              92|
+----+----------------+
only showing top 1 row

(3) Rata-rata rating pada masing-masing metode_pembayaran:
+-----------------+-----------------+
|metode_pembayaran|       avg rating|
+-----------------+-----------------+
|              COD|4.139442231075697|
|    Transfer Bank|4.130434782608695|
|         E-Wallet|            4.108|
|     Kartu Kredit|4.085365853658536|
+-----------------+-----------------+



In [7]:
#E

#Menyimpan DataFrame hasil olahan ke HDFS dalam format CSV
df.write.mode("overwrite").option("header", True).csv("hdfs://localhost:9000/user/mahasiswa/tugas4/hasil_transaksi_september_2026")

#Memastikan file berhasil disimpan
!hdfs dfs -ls /user/mahasiswa/tugas4/hasil_transaksi_september_2026

Found 2 items
-rw-r--r--   3 putrisenja supergroup          0 2026-09-16 08:16 /user/mahasiswa/tugas4/hasil_transaksi_september_2026/_SUCCESS
-rw-r--r--   3 putrisenja supergroup      92296 2026-09-16 08:16 /user/mahasiswa/tugas4/hasil_transaksi_september_2026/part-00000-1d45fb3d-2bc9-4371-9eb2-9a73f68df6d7-c000.csv


In [8]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
